In [141]:
library(data.table)
library(dplyr)

In [142]:
mydir = '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/review_files/ct_mean_expression_files/'
mean_expr_files = list.files(mydir)
length(mean_expr_files)
tail(mean_expr_files)

[1] 617

[1] "Treg_chr4.csv" "Treg_chr5.csv" "Treg_chr6.csv" "Treg_chr7.csv"
[5] "Treg_chr8.csv" "Treg_chr9.csv"

In [143]:
suppl_tables_dir = '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/ms_tables/'
df_common_ct = as.data.frame(fread(paste0(suppl_tables_dir,'all_eqtls_fdr_5pct.tsv')))
head(df_common_ct,2)

,gene,ACAT_p,top_MarkerID,top_pval,qvalue,celltype,ASDC_is_expressed,B_intermediate_is_expressed,B_memory_is_expressed,B_naive_is_expressed,⋯,ILC_is_egene,MAIT_is_egene,NK_is_egene,NK_CD56bright_is_egene,NK_Proliferating_is_egene,pDC_is_egene,Plasmablast_is_egene,Treg_is_egene,n_celltypes_expressed,n_celltypes_egene
,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<int>,<int>
1,ENSG00000020633,5.782049e-04,1:24970252:A:C,1.42751e-05,1.925410e-02,ASDC,TRUE,TRUE,TRUE,TRUE,⋯,FALSE,TRUE,TRUE,FALSE,FALSE,FALSE,TRUE,TRUE,28,14
2,ENSG00000049247,1.234790e-12,1:7913587:T:TTTTTTGTTGTTGTTG,6.48378e-15,2.927374e-10,ASDC,TRUE,FALSE,FALSE,FALSE,⋯,TRUE,TRUE,TRUE,TRUE,TRUE,FALSE,FALSE,TRUE,14,14


In [149]:
celltypes = unique(df_common_ct$celltype)
celltypes

[1] "ASDC"              "B_intermediate"    "B_memory"         
 [4] "B_naive"           "CD14_Mono"         "CD16_Mono"        
 [7] "CD4_CTL"           "CD4_Naive"         "CD4_Proliferating"
[10] "CD4_TCM"           "CD4_TEM"           "CD8_Naive"        
[13] "CD8_Proliferating" "CD8_TCM"           "CD8_TEM"          
[16] "cDC1"              "cDC2"              "dnT"              
[19] "gdT"               "HSPC"              "ILC"              
[22] "MAIT"              "NK"                "NK_CD56bright"    
[25] "NK_Proliferating"  "pDC"               "Plasmablast"      
[28] "Treg"

In [150]:
df_all = data.frame(gene='',chrom='1')
for (celltype in celltypes){
    df_list = list()
    for (chrom in 1:22){
        myfile = paste0(mydir,celltype,'_chr',chrom,'.csv')
        df = as.data.frame(t(read.csv(myfile,row.names=1)))
        df$chrom = chrom
        df$gene = rownames(df)
        df_list[[chrom]] = df
    }
    df_chrom_combine = as.data.frame(rbindlist(df_list))
    df_all = merge(df_all, df_chrom_combine, by=c('gene','chrom'),all = TRUE)
}
df_all = df_all[-1,]

In [151]:
head(df_all,2)

,gene,chrom,mean_ASDC,mean_B_intermediate,mean_B_memory,mean_B_naive,mean_CD14_Mono,mean_CD16_Mono,mean_CD4_CTL,mean_CD4_Naive,⋯,mean_gdT,mean_HSPC,mean_ILC,mean_MAIT,mean_NK,mean_NK_CD56bright,mean_NK_Proliferating,mean_pDC,mean_Plasmablast,mean_Treg
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2,ENSG00000000419,20,1.0882353,0.43761037,0.4225268,0.35185948,0.5324157,0.5811292,0.35086813,0.3855598,⋯,0.3944346,0.9328859,0.3084990,0.3769801,0.35631074,0.3969260,0.6485504,0.7954968,1.1469380,0.3598688
3,ENSG00000000457,1,0.1941176,0.08857861,0.1054823,0.08342382,0.1218999,0.1714356,0.09576565,0.1009841,⋯,0.1049358,0.2601342,0.1605855,0.1041616,0.08793871,0.1539688,0.1495168,0.4966949,0.2637029,0.1081986


In [152]:
dim(df_all)

[1] 37178    30

In [153]:
write.csv(df_all, paste0(mydir,'mean_expression_all_genes_all_celltypes.csv'))